# Microsoft agent-framework PyPI 1.26.1·1.26.2 악성 버전 정리: 내가 설치했는지 확인하는 법(lockfile·.pth 파일), 1.26.0 고정과 키 교체

위에서부터 차례로 실행해 보세요.

### 깨끗한 venv에 1.26.0 설치 후 버전 확인

In [ ]:
%%bash
set -eo pipefail
mkdir -p /content/lab && cd /content/lab
python3 -m venv v
. v/bin/activate
pip install -q "agent-framework==1.26.0" 2>&1 | tail -3
pip show agent-framework | head -3
ls v/lib/python3*/site-packages | grep -E '^agent_framework-1.26.0.dist-info'
echo '--- .pth in clean venv:'
find v -path '*site-packages/*' -name '*.pth' -type f
pip download agent-framework==1.26.0 --no-deps -d dl -q
sha256sum dl/*.whl

### 예제 lockfile 만들기

In [ ]:
%%bash
mkdir -p /content/proj/ci /content/proj/svc-b /content/proj/svc-c && cd /content/proj
printf 'requests==2.34.2\nagent-framework==1.26.1\n' > requirements.txt
printf 'agent_framework==1.26.2 ; python_version >= "3.10"\npytest==8.3.0\n' > ci/requirements-dev.txt
cat > svc-b/uv.lock <<'E'
version = 1

[[package]]
name = "agent-framework"
version = "1.26.1"
source = { registry = "https://pypi.org/simple" }

[[package]]
name = "requests"
version = "2.34.2"
E
cat > svc-c/poetry.lock <<'E'
[[package]]
name = "agent-framework"
version = "1.26.2"
description = "Microsoft Agent Framework for building AI Agents with Python."

[[package]]
name = "requests"
version = "2.34.2"
E
echo 'agent-framework==1.26.0' > svc-c/requirements-ok.txt

### lockfile grep으로 악성 버전 찾기

In [ ]:
%%bash
cd /content/proj
echo '=== 한 줄 grep (Endor Labs 원문 명령): .lock 은 놓친다'
grep -rnE 'agent[-_]framework.{0,20}1\.26\.[12]\b' --include='*.txt' --include='*.lock' --include='*.toml' --include='*.cfg' . || true
echo '=== 이름 줄 + 다음 2줄에서 version 찾기 (uv.lock, poetry.lock)'
grep -rn -A2 -E '^name = "agent[-_]framework"$' --include='*.lock' . | grep -E 'version = "1\.26\.[12]"'

### 무해한 .pth 가 python -c pass 만으로 실행됨

In [ ]:
%%bash
python3 -m venv /content/t
SP=$(ls -d /content/t/lib/python3*/site-packages)
echo '--- .pth 목록 (Python 없이 find)'
find /content/t -path '*site-packages/*' -name '*.pth' -type f
echo 'import sys; sys.stdout.write("[harmless-pth] executed at interpreter startup\n")' > $SP/zz_harmless_test.pth
find /content/t -path '*site-packages/*' -name '*.pth' -type f
echo '--- python -c pass'
/content/t/bin/python -c 'pass'
echo '--- python -S -c pass (site 생략)'
/content/t/bin/python -S -c 'pass'
rm $SP/zz_harmless_test.pth
echo '--- 삭제 후'
/content/t/bin/python -c 'pass'; echo done

### pip-tools 설치 후 해시 고정 requirements 만들기

In [ ]:
%%bash
set -eo pipefail
mkdir -p /content/h && cd /content/h
python3 -m venv b && . b/bin/activate
pip install -q pip-tools
echo 'agent-framework==1.26.0' > requirements.in
pip-compile --generate-hashes --strip-extras -o requirements.txt requirements.in 2>&1 | tail -2
grep -A2 '^agent-framework==' requirements.txt

### --require-hashes 통과, 해시 변조 시 거부

In [ ]:
%%bash
cd /content/h
python3 -m venv ok && ./ok/bin/pip install -q --require-hashes -r requirements.txt; echo "정상 exit=$?"
./ok/bin/pip list 2>/dev/null | grep -E '^agent-framework '
sed 's/--hash=sha256:14267ec8e6929c139015f8ec918adc638a71c7656901413426bb30ab199b6d71/--hash=sha256:0000000000000000000000000000000000000000000000000000000000000000/; s/--hash=sha256:b20870b77f237bcf52b6abbab86364f7fcbb23d41d217fd0d643556d19462167/--hash=sha256:1111111111111111111111111111111111111111111111111111111111111111/' requirements.txt > bad.txt
python3 -m venv bad
./bad/bin/pip install --require-hashes -r bad.txt 2>&1 | tail -8; echo "변조 exit=${PIPESTATUS[0]}"
./bad/bin/pip list 2>/dev/null | grep -c agent-framework || true